# Phase 2: prepare the data
What you're doing: fixing what you found in phase 1, and splitting the data into parts.

How to think about it: the whole point of a model is to work on wafers it has never seen. So you have to hide some wafers from it and only test on those at the end. Ask yourself:

In [1]:
# How much data is there, and what does one example look like?
import numpy as np

DEFECTS = ["Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc", "Near-Full", "Scratch", "Random"]

data = np.load("../data/Wafer_Map_Datasets.npz")
maps = data["arr_0"]  # wafer maps
labels = data["arr_1"]  # labels

#(count, width, height, data type)
#(38015, 52, 52, int32)
print("Map:  ", maps.shape, labels.dtype)

#(count, columns, data type)
#(38015, 8, int32)
print("labels:", labels.shape, labels.dtype)

Map:   (38015, 52, 52) int32
labels: (38015, 8) int32


# If my model has secretly seen a test wafer during training, what happens to my score? Is that score honest?
If the model trains on a wafer and then tests on the same wafer, it will memorize it. Your score will be artificially high and won't reflect real performance on unseen data. \
**The risk:** Your score looks great, but the model actually sucks on new wafers.

# Which of the things I found in phase 1 could cause that?

Look back at your EDA. What could cause data leakage? \
**Example:** If there are duplicate wafers (same pixel map twice), one could accidentally end up in training and one in testing. The model "sees" it twice. \
**Check:** Did you find any duplicate wafers? Any other issues that could leak data?


# Every class has to show up in every part. What if a class is small?
You need train/validation/test sets. Each set must have examples of ALL defect types. \
**Problem:** Near-Full has only 149 wafers. If you randomly split 38,015 wafers into 70/15/15, what if all 149 Near-Full wafers end up in training? Then you have zero Near-Full examples to test on. \
**Solution:** Stratified split (ensure each set gets a fair share of rare classes).

# Why do people use three parts instead of two? (Find out before you decide.)
**Two parts:** train/test. But how do you pick hyperparameters? \
**Three parts:** train/validation/test. Train on part 1, tune on part 2, test on part 3. This prevents overfitting.